In [173]:
!pip install nba_api

# Get a list of all current players

In [174]:
from nba_api.stats.static import players

all_players = players.get_players()
all_players = [player['id'] for player in all_players if (player['is_active'] == True)]
len(all_players), all_players[0]

(573, 1630173)

# Get seasons

In [175]:
from nba_api.stats.endpoints import commonplayerinfo

player_info = commonplayerinfo.CommonPlayerInfo(player_id=all_players[0])
player_df = player_info.available_seasons.get_data_frame()
player_df['SEASON_ID'] = player_df[player_df['SEASON_ID'].str.startswith('1')]
player_df.dropna(inplace=True)
seasons = player_df['SEASON_ID'].to_list()
player_df

,SEASON_ID
0,12020
3,12021
6,12022
9,12023


# Get all games played

In [176]:
import pandas as pd
from nba_api.stats.endpoints import playergamelog

all_games_df = pd.DataFrame()
# Define the list of seasons (last 10 seasons)
seasons = ['2023-24', '2022-23', '2021-22', '2020-21', '2019-20', 
           '2018-19', '2017-18', '2016-17', '2015-16', '2014-15']

# player_ids = player.get_players()
# player_ids = player_ids['PLAYER_ID']
# for playerId in player_ids:
for season in seasons:
    game_log = playergamelog.PlayerGameLog(player_id=2544, season=season)
    game_df = game_log.get_data_frames()[0]
    all_games_df = pd.concat([all_games_df, game_df], ignore_index=True)
    
all_games_df

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22023,2544,0022301195,"APR 14, 2024",LAL @ NOP,W,38,11,20,0.550,0,2,0.000,6,6,1.000,2,9,11,17,5,1,4,0,28,19,1
1,22023,2544,0022301177,"APR 12, 2024",LAL @ MEM,W,41,13,20,0.650,3,7,0.429,8,11,0.727,2,7,9,5,2,0,8,1,37,-3,1
2,22023,2544,0022301155,"APR 09, 2024",LAL vs. GSW,L,36,14,22,0.636,1,3,0.333,4,5,0.800,1,6,7,11,2,0,4,0,33,-6,1
3,22023,2544,0022301127,"APR 06, 2024",LAL vs. CLE,W,36,10,18,0.556,1,5,0.200,3,5,0.600,0,5,5,12,1,1,5,1,24,10,1
4,22023,2544,0022301103,"APR 03, 2024",LAL @ WAS,W,36,9,18,0.500,0,1,0.000,7,9,0.778,2,5,7,9,3,0,4,2,25,9,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,22014,2544,0021400081,"NOV 07, 2014",CLE @ DEN,W,40,8,18,0.444,1,5,0.200,5,6,0.833,0,7,7,11,1,1,3,1,22,14,1
646,22014,2544,0021400066,"NOV 05, 2014",CLE @ UTA,L,42,8,18,0.444,3,5,0.600,12,12,1.000,0,3,3,4,3,0,4,2,31,3,1
647,22014,2544,0021400055,"NOV 04, 2014",CLE @ POR,L,35,4,12,0.333,2,4,0.500,1,1,1.000,1,6,7,7,1,0,3,2,11,-15,1
648,22014,2544,0021400022,"OCT 31, 2014",CLE @ CHI,W,42,14,30,0.467,1,3,0.333,7,9,0.778,1,7,8,5,4,1,3,1,36,-3,1


In [177]:
pd.options.display.max_columns = None

all_games_df['YEAR'] = all_games_df['GAME_DATE'].str[-4:]
all_games_df['TEAM'] = all_games_df['MATCHUP'].str[:3]
all_games_df['RIVAL'] = all_games_df['MATCHUP'].str[-3:]
all_games_df['AWAY'] = all_games_df['MATCHUP'].str.contains("@")
all_games_df.drop(columns = 'MATCHUP', inplace=True)
all_games_df.head()

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,YEAR,TEAM,RIVAL,AWAY
0,22023,2544,0022301195,"APR 14, 2024",W,38,11,20,0.550,0,2,0.000,6,6,1.000,2,9,11,17,5,1,4,0,28,19,1,2024,LAL,NOP,True
1,22023,2544,0022301177,"APR 12, 2024",W,41,13,20,0.650,3,7,0.429,8,11,0.727,2,7,9,5,2,0,8,1,37,-3,1,2024,LAL,MEM,True
2,22023,2544,0022301155,"APR 09, 2024",L,36,14,22,0.636,1,3,0.333,4,5,0.800,1,6,7,11,2,0,4,0,33,-6,1,2024,LAL,GSW,False
3,22023,2544,0022301127,"APR 06, 2024",W,36,10,18,0.556,1,5,0.200,3,5,0.600,0,5,5,12,1,1,5,1,24,10,1,2024,LAL,CLE,False
4,22023,2544,0022301103,"APR 03, 2024",W,36,9,18,0.500,0,1,0.000,7,9,0.778,2,5,7,9,3,0,4,2,25,9,1,2024,LAL,WAS,True


# Get all team statistics for the last 10 seasons

In [178]:
from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats.static import teams
import pandas as pd
import time

# Get all NBA teams
all_teams = teams.get_teams()

# Define the list of seasons (last 10 seasons)
seasons = ['2023-24', '2022-23', '2021-22', '2020-21', '2019-20', 
           '2018-19', '2017-18', '2016-17', '2015-16', '2014-15']

# Empty list to store team stats
all_team_stats = []

# Loop through each season and get team stats
for season in seasons:
    print(f"Fetching data for season: {season}")
    
    # Fetching team stats for a given season
    stats = leaguedashteamstats.LeagueDashTeamStats(season=season)
    
    # Convert to DataFrame and add season column
    df = stats.get_data_frames()[0]
    df['SEASON'] = season
    
    # Append to list
    all_team_stats.append(df)
    
    # To avoid hitting API rate limits, let's pause for 1 second between requests
    time.sleep(1)

# Concatenate all season data into one DataFrame
final_df = pd.concat(all_team_stats, ignore_index=True)

# Display the final DataFrame
final_df

Fetching data for season: 2023-24
Fetching data for season: 2022-23
Fetching data for season: 2021-22
Fetching data for season: 2020-21
Fetching data for season: 2019-20
Fetching data for season: 2018-19
Fetching data for season: 2017-18
Fetching data for season: 2016-17
Fetching data for season: 2015-16
Fetching data for season: 2014-15


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,SEASON
0,1610612737,Atlanta Hawks,82,36,46,0.439,3971.0,3529,7584,0.465,1125,3092,0.364,1520,1906,0.797,1024,2639,3663,2180,1110.0,615,369,461,1522,1594,9703,-179.0,1,21,21,21,3,10,2,21,6,7,17,5,7,8,3,22,6,16,16,16,26,22,15,7,5,21,2023-24
1,1610612738,Boston Celtics,82,64,18,0.780,3966.0,3601,7396,0.487,1351,3482,0.388,1334,1654,0.807,876,2923,3799,2207,979.0,557,538,304,1326,1416,9887,930.0,1,1,1,1,7,4,8,8,1,1,2,19,26,7,14,1,2,14,1,27,1,1,2,30,2,1,2023-24
2,1610612751,Brooklyn Nets,82,32,50,0.390,3961.0,3334,7307,0.456,1089,3010,0.362,1293,1711,0.756,938,2675,3613,2102,1076.0,556,424,409,1516,1496,9050,-237.0,1,22,22,22,10,25,16,28,9,9,19,23,20,28,7,20,11,20,12,28,12,16,13,21,25,22,2023-24
3,1610612766,Charlotte Hornets,82,21,61,0.256,3946.0,3281,7133,0.460,989,2788,0.355,1189,1512,0.786,765,2538,3303,2033,1129.0,562,371,396,1472,1432,8740,-840.0,1,27,27,27,23,27,25,26,23,17,21,30,30,11,27,29,30,26,18,25,25,13,7,29,28,30,2023-24
4,1610612741,Chicago Bulls,82,39,43,0.476,3996.0,3448,7339,0.470,941,2630,0.358,1369,1731,0.791,916,2677,3593,2048,1004.0,638,394,404,1541,1538,9206,-118.0,1,20,20,20,1,16,13,18,27,26,20,17,18,10,8,19,14,23,3,9,18,14,17,14,22,20,2023-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,1610612758,Sacramento Kings,82,29,53,0.354,3971.0,3010,6617,0.455,461,1350,0.341,1829,2400,0.762,895,2728,3623,1667,1333.0,550,324,508,1696,1958,8310,-304.0,1,25,25,25,13,22,28,13,28,28,20,1,1,10,14,9,9,26,27,26,28,30,16,1,14,25,2014-15
296,1610612759,San Antonio Spurs,82,55,27,0.671,3991.0,3208,6854,0.468,677,1847,0.367,1368,1754,0.780,806,2772,3578,2000,1146.0,657,444,362,1564,1613,8461,508.0,1,5,5,5,1,4,11,3,12,15,6,17,23,4,27,8,15,5,13,11,8,9,8,19,7,3,2014-15
297,1610612761,Toronto Raptors,82,49,33,0.598,3971.0,3108,6829,0.455,726,2060,0.352,1585,2014,0.787,881,2526,3407,1701,1057.0,615,357,413,1712,1668,8527,252.0,1,11,11,11,13,11,14,12,8,9,11,4,7,2,15,27,26,22,3,18,23,19,18,13,4,9,2014-15
298,1610612762,Utah Jazz,82,38,44,0.463,3941.0,2900,6492,0.447,610,1781,0.343,1391,1929,0.721,988,2617,3605,1632,1256.0,623,489,382,1583,1604,7801,18.0,1,18,18,18,30,27,29,19,18,17,19,14,13,26,4,20,11,29,26,16,3,13,12,22,26,17,2014-15


In [179]:
final_df.columns

Index(['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA',
       'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB',
       'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS',
       'PLUS_MINUS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK',
       'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK',
       'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK',
       'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK',
       'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK',
       'SEASON'],
      dtype='object')

In [180]:
final_df.drop(columns=['GP', 'W', 'L', 'MIN', 'FGM', 'FGA',
       'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB',
       'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS',
       'PLUS_MINUS'], inplace=True)

In [181]:
team_abbreviation = all_games_df['TEAM'][2]
team_info = teams.find_team_by_abbreviation(team_abbreviation)
team_info

{'id': 1610612747,
 'full_name': 'Los Angeles Lakers',
 'abbreviation': 'LAL',
 'nickname': 'Lakers',
 'city': 'Los Angeles',
 'state': 'California',
 'year_founded': 1948}

In [182]:
id_list = []
for i in range(len(all_games_df)):
    team_abbreviation = all_games_df['TEAM'][i]
    team_info = teams.find_team_by_abbreviation(team_abbreviation)
    team_id = team_info['id']
    id_list.append(team_id)
len(id_list)

650

In [183]:
rival_id_list = []
for i in range(len(all_games_df)):
    rival_team_abbreviation = all_games_df['RIVAL'][i]
    rival_team_info = teams.find_team_by_abbreviation(rival_team_abbreviation)
    rival_team_id = rival_team_info['id']
    rival_id_list.append(rival_team_id)
len(rival_id_list)

650

In [184]:
all_games_df['TEAM_ID'] = id_list
all_games_df['RIVAL_ID'] = rival_id_list

In [185]:
seasons = final_df['SEASON'].to_list()
season_ids = [f"2{season[:4]}" for season in seasons]
final_df['SEASON_ID'] = season_ids

In [186]:
final_df

,TEAM_ID,TEAM_NAME,W_PCT,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,SEASON,SEASON_ID
0,1610612737,Atlanta Hawks,0.439,1,21,21,21,3,10,2,21,6,7,17,5,7,8,3,22,6,16,16,16,26,22,15,7,5,21,2023-24,22023
1,1610612738,Boston Celtics,0.780,1,1,1,1,7,4,8,8,1,1,2,19,26,7,14,1,2,14,1,27,1,1,2,30,2,1,2023-24,22023
2,1610612751,Brooklyn Nets,0.390,1,22,22,22,10,25,16,28,9,9,19,23,20,28,7,20,11,20,12,28,12,16,13,21,25,22,2023-24,22023
3,1610612766,Charlotte Hornets,0.256,1,27,27,27,23,27,25,26,23,17,21,30,30,11,27,29,30,26,18,25,25,13,7,29,28,30,2023-24,22023
4,1610612741,Chicago Bulls,0.476,1,20,20,20,1,16,13,18,27,26,20,17,18,10,8,19,14,23,3,9,18,14,17,14,22,20,2023-24,22023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,1610612758,Sacramento Kings,0.354,1,25,25,25,13,22,28,13,28,28,20,1,1,10,14,9,9,26,27,26,28,30,16,1,14,25,2014-15,22014
296,1610612759,San Antonio Spurs,0.671,1,5,5,5,1,4,11,3,12,15,6,17,23,4,27,8,15,5,13,11,8,9,8,19,7,3,2014-15,22014
297,1610612761,Toronto Raptors,0.598,1,11,11,11,13,11,14,12,8,9,11,4,7,2,15,27,26,22,3,18,23,19,18,13,4,9,2014-15,22014
298,1610612762,Utah Jazz,0.463,1,18,18,18,30,27,29,19,18,17,19,14,13,26,4,20,11,29,26,16,3,13,12,22,26,17,2014-15,22014


In [187]:
all_games_df

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,YEAR,TEAM,RIVAL,AWAY,TEAM_ID,RIVAL_ID
0,22023,2544,0022301195,"APR 14, 2024",W,38,11,20,0.550,0,2,0.000,6,6,1.000,2,9,11,17,5,1,4,0,28,19,1,2024,LAL,NOP,True,1610612747,1610612740
1,22023,2544,0022301177,"APR 12, 2024",W,41,13,20,0.650,3,7,0.429,8,11,0.727,2,7,9,5,2,0,8,1,37,-3,1,2024,LAL,MEM,True,1610612747,1610612763
2,22023,2544,0022301155,"APR 09, 2024",L,36,14,22,0.636,1,3,0.333,4,5,0.800,1,6,7,11,2,0,4,0,33,-6,1,2024,LAL,GSW,False,1610612747,1610612744
3,22023,2544,0022301127,"APR 06, 2024",W,36,10,18,0.556,1,5,0.200,3,5,0.600,0,5,5,12,1,1,5,1,24,10,1,2024,LAL,CLE,False,1610612747,1610612739
4,22023,2544,0022301103,"APR 03, 2024",W,36,9,18,0.500,0,1,0.000,7,9,0.778,2,5,7,9,3,0,4,2,25,9,1,2024,LAL,WAS,True,1610612747,1610612764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,22014,2544,0021400081,"NOV 07, 2014",W,40,8,18,0.444,1,5,0.200,5,6,0.833,0,7,7,11,1,1,3,1,22,14,1,2014,CLE,DEN,True,1610612739,1610612743
646,22014,2544,0021400066,"NOV 05, 2014",L,42,8,18,0.444,3,5,0.600,12,12,1.000,0,3,3,4,3,0,4,2,31,3,1,2014,CLE,UTA,True,1610612739,1610612762
647,22014,2544,0021400055,"NOV 04, 2014",L,35,4,12,0.333,2,4,0.500,1,1,1.000,1,6,7,7,1,0,3,2,11,-15,1,2014,CLE,POR,True,1610612739,1610612757
648,22014,2544,0021400022,"OCT 31, 2014",W,42,14,30,0.467,1,3,0.333,7,9,0.778,1,7,8,5,4,1,3,1,36,-3,1,2014,CLE,CHI,True,1610612739,1610612741


In [188]:
import pandas as pd

def add_team_row(df, df_to_add, team_id_col='TEAM_ID', season_id_col='SEASON_ID'):
    """
    Adds team-related columns from df_to_add to df with 'TEAM_' prefix.
    
    Parameters:
    - df: DataFrame to which columns will be added
    - df_to_add: DataFrame containing team-related data
    - team_id_col: Column name for team IDs
    - season_id_col: Column name for season IDs
    """
    # Rename columns in df_to_add by adding 'TEAM_' prefix
    df_to_add_renamed = df_to_add.rename(columns=lambda x: f"TEAM_{x}" if x not in [team_id_col, season_id_col] else x)
    
    # Merge the team data into the main DataFrame
    df = df.merge(df_to_add_renamed, how='left', left_on=[team_id_col, season_id_col], right_on=[team_id_col, season_id_col])
    
    return df

# Example structure of final_df (ensure this DataFrame has the correct columns)
# final_df = pd.DataFrame({
#     'TEAM_ID': [...],
#     'SEASON_ID': [...],
#     'TEAM_STAT_1': [...],
#     'TEAM_STAT_2': [...]
# })

# Add team data to all_games_df
all_games_df = add_team_row(all_games_df, final_df)

# Display the resulting DataFrame
all_games_df

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE,YEAR,TEAM,RIVAL,AWAY,TEAM_ID,RIVAL_ID,TEAM_TEAM_NAME,TEAM_W_PCT,TEAM_GP_RANK,TEAM_W_RANK,TEAM_L_RANK,TEAM_W_PCT_RANK,TEAM_MIN_RANK,TEAM_FGM_RANK,TEAM_FGA_RANK,TEAM_FG_PCT_RANK,TEAM_FG3M_RANK,TEAM_FG3A_RANK,TEAM_FG3_PCT_RANK,TEAM_FTM_RANK,TEAM_FTA_RANK,TEAM_FT_PCT_RANK,TEAM_OREB_RANK,TEAM_DREB_RANK,TEAM_REB_RANK,TEAM_AST_RANK,TEAM_TOV_RANK,TEAM_STL_RANK,TEAM_BLK_RANK,TEAM_BLKA_RANK,TEAM_PF_RANK,TEAM_PFD_RANK,TEAM_PTS_RANK,TEAM_PLUS_MINUS_RANK,TEAM_SEASON
0,22023,2544,0022301195,"APR 14, 2024",W,38,11,20,0.550,0,2,0.000,6,6,1.000,2,9,11,17,5,1,4,0,28,19,1,2024,LAL,NOP,True,1610612747,1610612740,Los Angeles Lakers,0.573,1,12,12,12,3,6,22,2,24,28,8,2,2,15,30,2,18,5,19,19,11,11,1,4,6,19,2023-24
1,22023,2544,0022301177,"APR 12, 2024",W,41,13,20,0.650,3,7,0.429,8,11,0.727,2,7,9,5,2,0,8,1,37,-3,1,2024,LAL,MEM,True,1610612747,1610612763,Los Angeles Lakers,0.573,1,12,12,12,3,6,22,2,24,28,8,2,2,15,30,2,18,5,19,19,11,11,1,4,6,19,2023-24
2,22023,2544,0022301155,"APR 09, 2024",L,36,14,22,0.636,1,3,0.333,4,5,0.800,1,6,7,11,2,0,4,0,33,-6,1,2024,LAL,GSW,False,1610612747,1610612744,Los Angeles Lakers,0.573,1,12,12,12,3,6,22,2,24,28,8,2,2,15,30,2,18,5,19,19,11,11,1,4,6,19,2023-24
3,22023,2544,0022301127,"APR 06, 2024",W,36,10,18,0.556,1,5,0.200,3,5,0.600,0,5,5,12,1,1,5,1,24,10,1,2024,LAL,CLE,False,1610612747,1610612739,Los Angeles Lakers,0.573,1,12,12,12,3,6,22,2,24,28,8,2,2,15,30,2,18,5,19,19,11,11,1,4,6,19,2023-24
4,22023,2544,0022301103,"APR 03, 2024",W,36,9,18,0.500,0,1,0.000,7,9,0.778,2,5,7,9,3,0,4,2,25,9,1,2024,LAL,WAS,True,1610612747,1610612764,Los Angeles Lakers,0.573,1,12,12,12,3,6,22,2,24,28,8,2,2,15,30,2,18,5,19,19,11,11,1,4,6,19,2023-24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
645,22014,2544,0021400081,"NOV 07, 2014",W,40,8,18,0.444,1,5,0.200,5,6,0.833,0,7,7,11,1,1,3,1,22,14,1,2014,CLE,DEN,True,1610612739,1610612743,Cleveland Cavaliers,0.646,1,7,7,7,23,15,24,8,4,2,5,9,12,18,11,21,18,10,17,19,25,10,5,11,8,5,2014-15
646,22014,2544,0021400066,"NOV 05, 2014",L,42,8,18,0.444,3,5,0.600,12,12,1.000,0,3,3,4,3,0,4,2,31,3,1,2014,CLE,UTA,True,1610612739,1610612762,Cleveland Cavaliers,0.646,1,7,7,7,23,15,24,8,4,2,5,9,12,18,11,21,18,10,17,19,25,10,5,11,8,5,2014-15
647,22014,2544,0021400055,"NOV 04, 2014",L,35,4,12,0.333,2,4,0.500,1,1,1.000,1,6,7,7,1,0,3,2,11,-15,1,2014,CLE,POR,True,1610612739,1610612757,Cleveland Cavaliers,0.646,1,7,7,7,23,15,24,8,4,2,5,9,12,18,11,21,18,10,17,19,25,10,5,11,8,5,2014-15
648,22014,2544,0021400022,"OCT 31, 2014",W,42,14,30,0.467,1,3,0.333,7,9,0.778,1,7,8,5,4,1,3,1,36,-3,1,2014,CLE,CHI,True,1610612739,1610612741,Cleveland Cavaliers,0.646,1,7,7,7,23,15,24,8,4,2,5,9,12,18,11,21,18,10,17,19,25,10,5,11,8,5,2014-15


In [189]:
def add_team_row(df, row):
    for col in df_to_add.columns:
        df[f"TEAM_{col}"] = df_to_add[col]
        
for i in range(len(all_games_df)):
    game_team_id = all_games_df['TEAM_ID'][i]
    season_id = all_games_df['SEASON_ID'][i]
    row = final_df[game_team_id][season_id]
    add_row(all_games_df, row)

KeyError: 1610612747

In [ ]:
all_games_df